In [ ]:
import pickle
import os
import numpy as np

In [ ]:
# Path to your saved pickle file
pickle_path = "../../data/Georgios/pcy_pass1/pcy_pass1_results.pkl"

# 1. Check if the file exists
if not os.path.exists(pickle_path):
    raise FileNotFoundError(f"Pickle file not found at: {pickle_path}")

print(f"Pickle found at: {pickle_path}")

# 2. Load pickle
with open(pickle_path, "rb") as f:
    data = pickle.load(f)

print("Pickle loaded successfully!")

# 3. Extract variables
baskets = data["baskets"]
num_unique_items = data["num_unique_items"]
bucket_options = data["bucket_options"]
support_percentages = data["support_percentages"]
results = data["results"]

# 4. Print confirmation
print("Loaded keys:", data.keys())
print("Number of baskets:", len(baskets))
print("Unique items:", num_unique_items)
print("Bucket options:", bucket_options)
print("Support percentages:", support_percentages)

print("\nMatrix results shape:", results.shape)
print("Everything is loaded and ready for the final PCY run.")


Pickle found at: ../../data/Georgios/pcy_pass1/pcy_pass1_results.pkl
Pickle loaded successfully!
Loaded keys: dict_keys(['baskets', 'num_unique_items', 'bucket_options', 'support_percentages', 'results'])
Number of baskets: 3214874
Unique items: 49677
Bucket options: [49677, 74515, 99354, 149031]
Support percentages: [0.001, 0.002, 0.005, 0.01]

Matrix results shape: (4, 4)
Everything is loaded and ready for the final PCY run.


In [ ]:
# final parameters
num_buckets = 75000                      # chosen from the experiment
support_percentage = 0.005               # 0.5%
support_threshold = int(len(baskets) * support_percentage)

print("Final PCY configuration:")
print(" - Buckets:", num_buckets)
print(" - Support threshold:", support_threshold)

# final pass 1
print("\nRunning PCY Final Pass 1...")

# Count single items
from collections import Counter
item_counts = Counter()
for basket in baskets:
    item_counts.update(basket)

# Bucket counts
bucket_counts = np.zeros(num_buckets, dtype=int)

# Hash pairs into buckets
for basket in baskets:
    b = sorted(basket)
    for i in range(len(b)):
        for j in range(i+1, len(b)):
            a, c = b[i], b[j]
            h = (a * 13 + c * 7) % num_buckets
            bucket_counts[h] += 1

print("Pass 1 complete!")

# Bitmaps
bitmap = (bucket_counts >= support_threshold)
print("Bitmap created.")
print("Number of frequent buckets:", bitmap.sum())

Final PCY configuration:
 - Buckets: 75000
 - Support threshold: 16074

Running PCY Final Pass 1...
Pass 1 complete!
Bitmap created.
Number of frequent buckets: 144


In [ ]:
from collections import defaultdict

print("Running PCY Final Pass 2...")

# ---- Step 1: filter frequent single items ----
frequent_items = {item for item, count in item_counts.items()
                    if count >= support_threshold}

print("Frequent single items:", len(frequent_items))

# ---- Step 2: count only promising pairs ----
pair_counts = defaultdict(int)

for basket in baskets:
    # keep only frequent items
    b = sorted([x for x in basket if x in frequent_items])
    
    # skip tiny baskets
    if len(b) < 2:
        continue
    
    # check pairs
    for i in range(len(b)):
        for j in range(i+1, len(b)):
            a, c = b[i], b[j]
            
            # compute hash bucket for this pair
            h = (a * 13 + c * 7) % num_buckets
            
            # only count pairs in frequent buckets
            if bitmap[h]:
                pair_counts[(a, c)] += 1

print("Pass 2 pair counting complete!")
print("Candidate pairs counted:", len(pair_counts))


Running PCY Final Pass 2...
Frequent single items: 252
Pass 2 pair counting complete!
Candidate pairs counted: 211


In [ ]:
# Extract FINAL frequent pairs
frequent_pairs = {pair: cnt for pair, cnt in pair_counts.items()
                    if cnt >= support_threshold}

print("True frequent pairs found:", len(frequent_pairs))


True frequent pairs found: 74
